<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/t05_MDanalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trajectory analysis with MDAnalysis

The aim of this experience is to learn how to extract information from a Molecular Dynamics (MD) trajectory. We will analyse two simulations of the C-terminal DNA-binding domain of HIV-1 integrase, one of the monomer and one of the dimer, and ask a simple question: **what changes when the protein forms a dimer?**

To do this we will use [MDAnalysis](https://www.mdanalysis.org), a Python library to read and analyse trajectories, and go from simple quantities (RMSD, RMSF) to the structure and the interactions of the protein (secondary structure, hydrogen bonds, the dimer interface) and finally to the conformational space it explores (clustering and principal component analysis).

The reference lecture is:

*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/03_MolecularDynamics.pdf

## Preliminary setup

We need MDAnalysis for the analysis and py3Dmol to look at the structures in the notebook. This analysis does not need a GPU.

In [ ]:
%pip install -q MDAnalysis py3Dmol

We work in a folder of the virtual machine (set `use_drive = True` to work in your Google Drive instead). The cell can be run more than once.

In [ ]:
import os
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()

use_drive = False        # True: keep your files on Google Drive (Colab will ask your permission)
if use_drive:
    from google.colab import drive
    drive.mount('/content/drive')
    workdir = '/content/drive/MyDrive/Structural_Bioinformatics/T05'
else:
    workdir = os.path.join(BASE_DIR, 'T05')
os.makedirs(workdir, exist_ok=True)
%cd $workdir

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # hide some harmless warnings of MDAnalysis

import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
import py3Dmol
print('MDAnalysis version', mda.__version__)

## Theoretical aspects

### Root Mean Square Deviation (RMSD)

The RMSD measures the deviation of a set of coordinates (a structure) from a reference set of coordinates, with RMSD = 0 indicating a perfect overlap. Along an MD trajectory, a low RMSD from the starting structure means that the protein does not change much on the time scale of the simulation.

$$RMSD = \sqrt{\frac{\sum_{i=1}^N m_i\,(\mathbf{X}_i - \mathbf{Y}_i)^2}{M}}$$

where $N$ is the number of atoms, $m_i$ the mass of atom $i$, $\mathbf{X}_i$ the coordinates of atom $i$ in the target structure, $\mathbf{Y}_i$ those in the reference, and $M$ the total mass. If the RMSD is not mass-weighted, $m_i = 1$ for all atoms and $M = N$.

The RMSD is usually calculated after the best superposition (rotation and translation) of the target on the reference. Two requirements follow, as we will see in the practical part:

1. the number of atoms in the target must match the number of atoms in the reference;
2. the atoms must be in the same order in the target and in the reference.

### Root Mean Square Fluctuation (RMSF)

The RMSF of atom $i$ measures how much its position fluctuates around its average position along the trajectory:

$$\rho_i = \sqrt{\left\langle (\mathbf{x}_i - \langle \mathbf{x}_i \rangle)^2 \right\rangle}$$

where $\langle \cdot \rangle$ is the average over the frames. While the RMSD tells how far the whole structure goes from a reference *as a function of time*, the RMSF tells *which parts* of the protein are the most mobile. For proteins it is usually calculated on the Cα atoms (or the backbone), which describe the conformational changes better than the side chains. As for the RMSD, the frames must first be superimposed, otherwise the RMSF would include the rotation and translation of the whole protein.

<figure>
<center>
<img src='https://www.frontiersin.org/files/Articles/329304/fphar-09-00492-HTML/image_m/fphar-09-00492-g002.jpg'/>
<figcaption>RMSD and RMSF plots of the structural changes occurring due to antagonist binding in the ligand binding pocket of androgen receptor, elucidated through MD simulations.<br>Sugunadevi S et al (2018)<i> Front Pharmacology 9, 492</i> </figcaption></center>
</figure>

### MDAnalysis in a nutshell

MDAnalysis is object oriented: atoms, groups of atoms and trajectories are Python objects, and each object has *attributes* (values that describe it, e.g. `atoms.names`, `atoms.positions`) and *methods* (operations, e.g. `atoms.center_of_mass()`). Three concepts are enough to start:

1.   **Universe**: everything starts by loading a topology (the list of atoms, from a `.gro`, `.pdb`, `.tpr`, ... file) together with a trajectory (the coordinates of each frame, from an `.xtc`, `.dcd`, ... file) into a `Universe`.
2.   **AtomGroup**: any group of atoms, for example `u.atoms` (all atoms). Its positions are those of the *current* frame of the trajectory.
3.   **Selections**: `u.select_atoms('...')` returns an AtomGroup selected with a simple language, e.g. `'name CA'`, `'resid 5-100'`, `'backbone'`, `'resname ASP GLU and name CA'`, combined with `and`, `or`, `not`. All the keywords are described in the [documentation](https://userguide.mdanalysis.org/stable/selections.html).

A typical analysis loops over the frames of the trajectory and, for each frame, computes something from the positions of an AtomGroup. MDAnalysis also contains many ready-made analyses (in `MDAnalysis.analysis`), which we will use below; they all work in the same way: you create the analysis object and call its `.run()` method, and the results are stored in `.results`.

## Part 0 – The system and the trajectories

The C-terminal domain of HIV-1 integrase (residues 220–270) binds DNA in a non-specific way. Its structure was solved by NMR ([PDB 1IHV](https://www.rcsb.org/structure/1IHV)) and shows a small β-barrel similar to an SH3 domain; in solution the domain forms a dimer.

We will analyse two simulations of 20 ns each, one of a single chain (monomer) and one of the dimer. The water and the ions were removed from the trajectories, which contain the protein only. Both trajectories have 1001 frames, saved every 20 ps.

The trajectories are part of the course [IBM3202](https://github.com/pb3lab/ibm3202) by the PB3 lab.

In [ ]:
# download the trajectories and the starting structures (only if they are not already in the folder)
import urllib.request
for name in ['1ihv_mon_protPBC.gro', '1ihv_mon_protPBC.xtc', '1ihv_dimer_protPBC.gro', '1ihv_dimer_protPBC.xtc']:
    if not os.path.exists(name):
        urllib.request.urlretrieve(f'https://github.com/pb3lab/ibm3202/raw/master/files/md_files/{name}', name)
    print(f'{name:<26}{os.path.getsize(name) / 1e6:6.2f} MB')

We load each simulation in a `Universe`. With `in_memory=True` the whole trajectory is read into memory: this makes the analyses faster and, as we will see, allows us to modify the coordinates (for example to superimpose all the frames).

In [ ]:
mon = mda.Universe('1ihv_mon_protPBC.gro', '1ihv_mon_protPBC.xtc', in_memory=True)
dim = mda.Universe('1ihv_dimer_protPBC.gro', '1ihv_dimer_protPBC.xtc', in_memory=True)

for name, u in [('monomer', mon), ('dimer', dim)]:
    print(f'{name}: {u.atoms.n_atoms} atoms, {u.residues.n_residues} residues, '
          f'{u.trajectory.n_frames} frames, one every {u.trajectory.dt:.0f} ps, '
          f'{u.trajectory.totaltime / 1000:.0f} ns in total')

Note that the times in MDAnalysis are in **ps** and the distances in **Å** (GROMACS uses nm).

### The two chains of the dimer

A `.gro` file does not store which chain each atom belongs to, and both chains of the dimer are numbered from 219 to 270. A selection like `resid 227-240` would therefore select the same residues *in both chains*. Since the two chains have the same number of atoms and chain B follows chain A, we can add a chain identifier ourselves:

In [ ]:
dim.add_TopologyAttr('chainID')
half = dim.atoms.n_atoms // 2
dim.atoms[:half].chainIDs = 'A'
dim.atoms[half:].chainIDs = 'B'

print(dim.select_atoms('resid 227-240 and name CA').n_atoms, 'CA atoms in resid 227-240')
print(dim.select_atoms('chainID A and resid 227-240 and name CA').n_atoms, 'CA atoms in resid 227-240 of chain A')

We can write the first frame of the dimer as a PDB file (which, unlike the `.gro`, keeps the chain identifiers) and look at it:

In [ ]:
dim.trajectory[0]
dim.atoms.write('dimer_start.pdb')

view = py3Dmol.view(width=700, height=450)
view.addModel(open('dimer_start.pdb').read(), 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'purple'}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'orange'}})
view.zoomTo()
view.show()

### Selections

Some examples of selections on the monomer. An AtomGroup can be selected further, and from an AtomGroup you can get its residues:

In [ ]:
ca = mon.select_atoms('protein and name CA')
print(ca)

acidic = ca.select_atoms('resname ASP GLU')
print(acidic)
print(list(acidic.residues))

print(mon.select_atoms('resid 230-240').n_residues, 'residues in resid 230-240 (the range includes both ends)')

**QUESTION❓:** how many positively charged residues (Lys and Arg) does the domain have? Which part of the protein would you expect to bind the (negatively charged) DNA?

In [ ]:
# your code here

### Writing your own analysis: radius of gyration and end-to-end distance

The basic pattern of an analysis is a loop over the frames of the trajectory. At each step of the loop `u.trajectory` moves to the next frame and the positions of all the AtomGroups are updated. Here we compute the radius of gyration of the protein, a measure of its size, and the distance between the N and the C terminus.

In [ ]:
protein = mon.select_atoms('protein')
n_term = protein.select_atoms('name N')[0]     # the first N atom
c_term = protein.select_atoms('name C')[-1]    # the last C atom

time, rgyr, end_to_end = [], [], []
for ts in mon.trajectory:
    time.append(ts.time / 1000)                 # ps -> ns
    rgyr.append(protein.radius_of_gyration())
    end_to_end.append(np.linalg.norm(c_term.position - n_term.position))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(time, rgyr)
axes[0].set_ylabel('Radius of gyration (Å)')
axes[1].plot(time, end_to_end)
axes[1].set_ylabel('N-C terminus distance (Å)')
for ax in axes:
    ax.set_xlabel('Time (ns)')
plt.show()

**QUESTION❓:** which of the two quantities fluctuates more? Why?

**Try it:** compute the radius of gyration of the dimer and of each of its chains (`chainID A`, `chainID B`) and compare it with that of the monomer.

In [ ]:
# your code here

## Part I – RMSD and RMSF

In [ ]:
from MDAnalysis.analysis import align, rms

### I.1 – RMSD

#### I.1A – Superposition

Let's compute the RMSD of the Cα atoms between the last and the first frame of the monomer simulation, first without superimposing the two structures:

In [ ]:
mobile = mda.Universe('1ihv_mon_protPBC.gro', '1ihv_mon_protPBC.xtc', in_memory=True)  # a copy of the monomer trajectory
ref = mda.Universe('1ihv_mon_protPBC.gro')       # the starting structure

mobile.trajectory[-1]                            # go to the last frame
mobile_ca = mobile.select_atoms('name CA')
ref_ca = ref.select_atoms('name CA')
rms.rmsd(mobile_ca.positions, ref_ca.positions, superposition=False)

`align.alignto` superimposes a single structure, while `align.AlignTraj` superimposes all the frames of a trajectory on a reference. With `in_memory=True` the superimposed coordinates replace the original ones in the Universe.

In [ ]:
align.AlignTraj(mobile, ref, select='name CA', in_memory=True).run()

mobile.trajectory[-1]
rms.rmsd(mobile_ca.positions, ref_ca.positions, superposition=False)

**QUESTION❓:** how do the two values compare? Which one is meaningful, and why?

#### I.1B – RMSD along the trajectory

The class `rms.RMSD` computes the RMSD of every frame from a reference. It first superimposes each frame on the reference using the atoms in `select` and computes their RMSD; then, *without further superposition*, it computes the RMSD of each group in `groupselections`. Here we follow the whole backbone and two regions of the protein:

In [ ]:
ref_mon = mda.Universe('1ihv_mon_protPBC.gro')   # starting structure of the monomer

loop1 = 'backbone and resid 227-240'
loop2 = 'backbone and resid 252-257'

rmsd_mon = rms.RMSD(mon, ref_mon, select='backbone', groupselections=[loop1, loop2]).run()
rmsd_mon.results.rmsd.shape

`rmsd_mon.results.rmsd` has a row for each frame. The columns are: the frame number, the time (ps), the RMSD of `select` and then the RMSD of each group in `groupselections`.

In [ ]:
r = rmsd_mon.results.rmsd
plt.figure(figsize=(10, 5))
plt.plot(r[:, 1] / 1000, r[:, 2], label='backbone')
plt.plot(r[:, 1] / 1000, r[:, 3], label='Loop 1 (227-240)')
plt.plot(r[:, 1] / 1000, r[:, 4], label='Loop 2 (252-257)')
plt.xlabel('Time (ns)')
plt.ylabel('RMSD (Å)')
plt.title('Monomer')
plt.legend()
plt.show()

**QUESTION❓:** what is the range of the RMSD fluctuations? Does the RMSD reach a plateau? Which region deviates more from the starting structure?

#### I.1C – RMSD of the dimer

For the dimer we can superimpose either the whole dimer or one chain at a time. The difference between the two tells us whether the chains move *with respect to each other*: if each chain keeps its structure but the two chains rotate a little with respect to each other, the RMSD of each chain stays low while that of the whole dimer increases.

In [ ]:
ref_dim = mda.Universe('1ihv_dimer_protPBC.gro')
ref_dim.add_TopologyAttr('chainID')
ref_dim.atoms[:half].chainIDs = 'A'
ref_dim.atoms[half:].chainIDs = 'B'

rmsd_dim = rms.RMSD(dim, ref_dim, select='backbone').run()
rmsd_A = rms.RMSD(dim, ref_dim, select='backbone and chainID A',
                  groupselections=['backbone and chainID A and resid 227-240',
                                   'backbone and chainID A and resid 252-257']).run()
rmsd_B = rms.RMSD(dim, ref_dim, select='backbone and chainID B').run()

plt.figure(figsize=(10, 5))
plt.plot(r[:, 1] / 1000, r[:, 2], label='monomer')
plt.plot(rmsd_dim.results.rmsd[:, 1] / 1000, rmsd_dim.results.rmsd[:, 2], label='dimer, whole')
plt.plot(rmsd_A.results.rmsd[:, 1] / 1000, rmsd_A.results.rmsd[:, 2], label='dimer, chain A')
plt.plot(rmsd_B.results.rmsd[:, 1] / 1000, rmsd_B.results.rmsd[:, 2], label='dimer, chain B')
plt.xlabel('Time (ns)')
plt.ylabel('Backbone RMSD (Å)')
plt.legend()
plt.show()

**QUESTIONS❓**

1. Is the structure of a single chain more stable in the monomer or in the dimer?
2. Do the two chains of the dimer move with respect to each other?

**Try it:** compare the RMSD of the two loops in the monomer and in the two chains of the dimer (the RMSD of the loops of chain A is already in `rmsd_A`; you need to compute that of chain B).

In [ ]:
# your code here

### I.2 – RMSF

To compute the fluctuations, all the frames must first be superimposed. We superimpose each trajectory on its starting structure using the Cα atoms; since the Universes were loaded `in_memory`, the superimposed coordinates replace the original ones and are used in all the following analyses.

In [ ]:
align.AlignTraj(mon, ref_mon, select='name CA', in_memory=True).run()
align.AlignTraj(dim, ref_dim, select='name CA', in_memory=True).run()

ca_mon = mon.select_atoms('name CA')
ca_A = dim.select_atoms('chainID A and name CA')
ca_B = dim.select_atoms('chainID B and name CA')

rmsf_mon = rms.RMSF(ca_mon).run().results.rmsf
rmsf_A = rms.RMSF(ca_A).run().results.rmsf
rmsf_B = rms.RMSF(ca_B).run().results.rmsf

plt.figure(figsize=(10, 5))
plt.plot(ca_mon.resids, rmsf_mon, label='monomer')
plt.plot(ca_A.resids, rmsf_A, label='dimer, chain A')
plt.plot(ca_B.resids, rmsf_B, label='dimer, chain B')
plt.xlabel('Residue number')
plt.ylabel('RMSF (Å)')
plt.legend()
plt.show()

**QUESTIONS❓**

1. Is there any difference in the local fluctuations between the monomer and the dimer?
2. Which regions show the highest fluctuations? What are their structural features (look at the structure above and at the secondary structure in Part II)?
3. The dimer was superimposed as a whole, so the RMSF of each chain also contains the motion of one chain with respect to the other. **Try it:** superimpose the dimer on chain A only (`select='chainID A and name CA'`) in a copy of the trajectory and recompute the RMSF of chain A. Does it change?

In [ ]:
# your code here

## Part II – Structure and interactions

RMSD and RMSF tell *how much* the protein moves. To understand *why*, we look at the structural elements and the interactions that hold the protein together.

### II.1 – Secondary structure

DSSP assigns the secondary structure of each residue from the pattern of backbone hydrogen bonds. MDAnalysis contains its own implementation, which assigns three classes: `H` (helix), `E` (β strand) and `-` (loop). DSSP needs the backbone atoms N, CA, C and O of every residue; the last residue of a chain has two terminal oxygens (`OC1`, `OC2`) instead of `O`, so we exclude it with a small function.

In [ ]:
from MDAnalysis.analysis.dssp import DSSP
from matplotlib.colors import ListedColormap

def dssp_atoms(atomgroup):
    """The atoms of the residues that have all the backbone atoms needed by DSSP."""
    ok = [set(['N', 'CA', 'C', 'O']) <= set(res.atoms.names) for res in atomgroup.residues]
    return atomgroup.residues[ok].atoms

ss_mon = DSSP(dssp_atoms(mon.select_atoms('protein'))).run()
ss_dim = DSSP(dssp_atoms(dim.select_atoms('protein'))).run()

print('residues analysed:', ss_mon.results.resids.shape[0], 'in the monomer,', ss_dim.results.resids.shape[0], 'in the dimer')
print('first frame of the monomer:')
print(''.join(ss_mon.results.dssp[0]))

`results.dssp` is an array of letters with one row per frame and one column per residue. To plot it as a map we turn each letter into a number:

In [ ]:
codes = {'-': 0, 'H': 1, 'E': 2}
cmap = ListedColormap(['white', 'tab:red', 'tab:blue'])

def plot_dssp(ax, dssp, resids, times, title):
    numbers = np.vectorize(codes.get)(dssp).T          # residues x frames
    ax.imshow(numbers, aspect='auto', cmap=cmap, vmin=0, vmax=2, interpolation='none', origin='lower',
              extent=[times[0], times[-1], resids[0] - 0.5, resids[-1] + 0.5])
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Residue number')
    ax.set_title(title)

times = np.array([ts.time / 1000 for ts in mon.trajectory])
n = ss_mon.results.resids.shape[0]          # residues of one chain

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
plot_dssp(axes[0], ss_mon.results.dssp, ss_mon.results.resids, times, 'monomer')
plot_dssp(axes[1], ss_dim.results.dssp[:, :n], ss_dim.results.resids[:n], times, 'dimer, chain A')
plot_dssp(axes[2], ss_dim.results.dssp[:, n:], ss_dim.results.resids[n:], times, 'dimer, chain B')
plt.show()
print('blue: β strand, red: helix, white: loop')

A more compact way to compare the simulations is the fraction of frames in which each residue is in a β strand:

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ss_mon.results.resids, (ss_mon.results.dssp == 'E').mean(axis=0), 'o-', label='monomer')
plt.plot(ss_dim.results.resids[:n], (ss_dim.results.dssp[:, :n] == 'E').mean(axis=0), 'o-', label='dimer, chain A')
plt.plot(ss_dim.results.resids[n:], (ss_dim.results.dssp[:, n:] == 'E').mean(axis=0), 'o-', label='dimer, chain B')
plt.xlabel('Residue number')
plt.ylabel('β strand fraction')
plt.legend()
plt.show()

**QUESTIONS❓**

1. How many strands does the domain have? Is the secondary structure stable along the simulations?
2. Compare with the RMSF: do the most flexible regions correspond to loops? Are there strands whose ends fray, i.e. are only partially formed?
3. Is there any difference between the monomer and the chains of the dimer?

### II.2 – Hydrogen bonds

A hydrogen bond is formed when a hydrogen bound to a donor atom (here N or O) points to an acceptor atom (here O). `HydrogenBondAnalysis` finds them using two geometric criteria: the donor-acceptor distance (≤ 3.0 Å by default) and the donor-hydrogen-acceptor angle (≥ 150°).

Our files do not contain the chemical bonds, so we specify the atoms by name: MDAnalysis then assigns each hydrogen to the donor closest to it. We start with the backbone hydrogen bonds (N-H···O=C), those that form the secondary structure:

In [ ]:
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis

hb_mon = HydrogenBondAnalysis(mon, donors_sel='name N', hydrogens_sel='name H', acceptors_sel='name O').run()

plt.figure(figsize=(10, 4))
plt.plot(times, hb_mon.count_by_time())
plt.xlabel('Time (ns)')
plt.ylabel('Number of backbone H-bonds')
plt.title('Monomer')
plt.show()

`results.hbonds` lists every hydrogen bond found, one per row: frame, donor, hydrogen and acceptor atom indices, distance and angle. From it we compute the *occupancy* of each hydrogen bond between a pair of residues, i.e. the fraction of the frames in which the donor residue and the acceptor residue form at least one hydrogen bond:

In [ ]:
def label(atom):
    """e.g. A:GLN252 (the chain is shown only if the Universe has chain identifiers)."""
    chain = f'{atom.chainID}:' if hasattr(atom, 'chainID') else ''
    return f'{chain}{atom.resname}{atom.resid}'

def occupancy_table(hb, u, n_max=15):
    """Print the n_max pairs of residues (donor, acceptor) with the highest hydrogen bond occupancy."""
    frames = {}
    for row in hb.results.hbonds:
        donor, acceptor = u.atoms[int(row[1])], u.atoms[int(row[3])]
        frames.setdefault((label(donor), label(acceptor)), set()).add(int(row[0]))
    table = sorted(((len(f) / hb.n_frames, pair) for pair, f in frames.items()), reverse=True)
    print(f'{"donor":<20}{"acceptor":<20}{"occupancy":>10}')
    for occupancy, (donor, acceptor) in table[:n_max]:
        print(f'{donor:<20}{acceptor:<20}{occupancy:>10.2f}')

occupancy_table(hb_mon, mon)

**QUESTIONS❓**

1. Which residues form the most stable hydrogen bonds? Are they in the strands, i.e. do they hold the β-sheet together?
2. Are there hydrogen bonds with a low occupancy in the regions with a high RMSF?

We can also look only at the hydrogen bonds *between* the two chains of the dimer, using the `between` option. Here we include the side chains: donors are all the N and O atoms, acceptors all the O atoms.

In [ ]:
hb_AB = HydrogenBondAnalysis(dim, donors_sel='name N* O*', hydrogens_sel='name H*', acceptors_sel='name O*',
                             between=['chainID A', 'chainID B']).run()

plt.figure(figsize=(10, 4))
plt.plot(times, hb_AB.count_by_time())
plt.xlabel('Time (ns)')
plt.ylabel('Number of H-bonds between the chains')
plt.show()

occupancy_table(hb_AB, dim)

**QUESTION❓:** are the hydrogen bonds between the chains many and stable? What other interactions could hold the dimer together? Remember that the water was removed from the trajectory, so hydrogen bonds mediated by water molecules cannot be seen.

### II.3 – The dimer interface

Let's characterise the interface with a simpler criterion: two residues are in *contact* when any of their heavy (non-hydrogen) atoms are closer than 4.5 Å. First, we follow the number of atom pairs in contact and the distance between the centres of the two chains along the simulation:

In [ ]:
from MDAnalysis.analysis import distances

heavy_A = dim.select_atoms('chainID A and not name H*')
heavy_B = dim.select_atoms('chainID B and not name H*')
cutoff = 4.5

n_contacts, centre_distance = [], []
for ts in dim.trajectory:
    pairs = distances.capped_distance(heavy_A.positions, heavy_B.positions, max_cutoff=cutoff,
                                      box=dim.dimensions, return_distances=False)
    n_contacts.append(len(pairs))
    centre_distance.append(np.linalg.norm(heavy_A.center_of_geometry() - heavy_B.center_of_geometry()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(times, n_contacts)
axes[0].set_ylabel(f'Atom pairs closer than {cutoff} Å')
axes[1].plot(times, centre_distance)
axes[1].set_ylabel('Distance between the chain centres (Å)')
for ax in axes:
    ax.set_xlabel('Time (ns)')
plt.show()

**QUESTION❓:** is the dimer stable during the simulation?

Now we find *which* residues form the interface: for each residue, we compute the fraction of frames in which it is in contact with the other chain.

In [ ]:
def contact_frequency(u, sel_1, sel_2, cutoff=4.5):
    """For each residue of sel_1, the fraction of frames in which any of its heavy atoms
    is closer than cutoff to a heavy atom of sel_2."""
    group_1 = u.select_atoms(f'({sel_1}) and not name H*')
    group_2 = u.select_atoms(f'({sel_2}) and not name H*')
    residue_of_atom = np.searchsorted(group_1.residues.resindices, group_1.resindices)
    counts = np.zeros(group_1.n_residues)
    for ts in u.trajectory:
        pairs = distances.capped_distance(group_1.positions, group_2.positions, max_cutoff=cutoff,
                                          box=u.dimensions, return_distances=False)
        counts[np.unique(residue_of_atom[pairs[:, 0]])] += 1
    return group_1.residues, counts / u.trajectory.n_frames

res_A, freq_A = contact_frequency(dim, 'chainID A', 'chainID B')
res_B, freq_B = contact_frequency(dim, 'chainID B', 'chainID A')

plt.figure(figsize=(10, 4))
plt.bar(res_A.resids - 0.2, freq_A, width=0.4, label='chain A')
plt.bar(res_B.resids + 0.2, freq_B, width=0.4, label='chain B')
plt.xlabel('Residue number')
plt.ylabel('Contact frequency with the other chain')
plt.legend()
plt.show()

interface = res_A[freq_A > 0.5]
print('Residues of chain A in contact with chain B for more than half of the simulation:')
print(' '.join(f'{r.resname}{r.resid}' for r in interface))

Let's see where the interface residues are (chain A in purple, with the interface residues as sticks, chain B in orange):

In [ ]:
view = py3Dmol.view(width=700, height=450)
view.addModel(open('dimer_start.pdb').read(), 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'purple'}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'orange'}})
view.addStyle({'chain': 'A', 'resi': [int(r) for r in interface.resids]}, {'stick': {'colorscheme': 'purpleCarbon'}})
view.zoomTo()
view.show()

Finally, we can connect the interface with the fluctuations: does a residue fluctuate less when it is at the interface?

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(res_A.resids, freq_A, color='lightgray', label='contact frequency (chain A)')
ax1.set_xlabel('Residue number')
ax1.set_ylabel('Contact frequency')
ax2 = ax1.twinx()
ax2.plot(ca_A.resids, rmsf_A - rmsf_mon, 'o-', color='tab:red', label='RMSF dimer A - RMSF monomer')
ax2.axhline(0, color='tab:red', lw=0.5)
ax2.set_ylabel('RMSF difference (Å)', color='tab:red')
plt.show()

**QUESTIONS❓**

1. Which residues form the interface? Which secondary structure elements do they belong to?
2. Do the interface residues fluctuate less in the dimer than in the monomer? Do the fluctuations change also far from the interface?
3. Is the interface symmetric, i.e. do the same residues of the two chains make contacts?

## Part III – Conformational space

So far we have looked at averages and at quantities as a function of time. Now we ask which *conformations* the protein visits and how they are related.

### III.1 – Pairwise RMSD

Instead of computing the RMSD of all frames from a single reference, we can compute it between all the pairs of frames. `DistanceMatrix` does not superimpose the frames, but we already superimposed the trajectories in section I.2. The result is a matrix in which element $(i, j)$ is the RMSD between frame $i$ and frame $j$ (after superposition). Blocks of low RMSD along the diagonal correspond to groups of similar conformations visited consecutively, while low values far from the diagonal mean that the protein comes back to a conformation it had visited before.

In [ ]:
from MDAnalysis.analysis import diffusionmap

matrix_mon = diffusionmap.DistanceMatrix(mon, select='name CA').run().results.dist_matrix
matrix_dim = diffusionmap.DistanceMatrix(dim, select='name CA').run().results.dist_matrix
print(matrix_mon.shape, matrix_dim.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, matrix, title in [(axes[0], matrix_mon, 'monomer'), (axes[1], matrix_dim, 'dimer')]:
    im = ax.imshow(matrix, cmap='viridis', origin='lower', extent=[times[0], times[-1], times[0], times[-1]])
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Time (ns)')
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label='RMSD (Å)')
plt.show()

**QUESTION❓:** can you comment on what these matrices show? Does the protein visit one or more groups of conformations? Does it come back to conformations visited earlier?

### III.2 – Clustering

*Clustering* groups the frames into sets of similar conformations (clusters). We use hierarchical clustering from the `scipy` library, starting from the pairwise RMSD matrix: first each frame is a cluster of its own, then the two closest clusters are merged, and so on. Stopping when the clusters are farther than a threshold gives clusters whose frames are, on average, within that RMSD from each other.

For each cluster we choose as *representative* the frame with the lowest average RMSD from the other frames of the cluster (the *medoid*).

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

def cluster_frames(matrix, threshold):
    """Hierarchical clustering (average linkage) of the frames from an RMSD matrix.
    Returns the cluster of each frame (0 = most populated cluster) and the medoid of each cluster."""
    tree = linkage(squareform(matrix, checks=False), method='average')
    labels = fcluster(tree, t=threshold, criterion='distance')
    # renumber the clusters from the most to the least populated
    order = np.argsort(-np.bincount(labels)[1:]) + 1
    labels = np.argsort(order)[labels - 1]
    medoids = []
    for c in range(labels.max() + 1):
        members = np.where(labels == c)[0]
        medoids.append(members[np.argmin(matrix[np.ix_(members, members)].mean(axis=1))])
    return labels, medoids

threshold = 2.0   # Å
labels_dim, medoids_dim = cluster_frames(matrix_dim, threshold)

print(f'{labels_dim.max() + 1} clusters with a threshold of {threshold} Å')
for c, m in enumerate(medoids_dim[:5]):
    print(f'cluster {c}: {np.mean(labels_dim == c) * 100:5.1f}% of the frames, representative frame {m} ({times[m]:.2f} ns)')

plt.figure(figsize=(10, 3))
plt.plot(times, labels_dim, '.')
plt.xlabel('Time (ns)')
plt.ylabel('Cluster')
plt.title('Dimer')
plt.show()

Let's superimpose the representatives of the three most populated clusters:

In [ ]:
colors = ['purple', 'orange', 'green']
view = py3Dmol.view(width=700, height=450)
for c, m in enumerate(medoids_dim[:3]):
    dim.trajectory[m]
    dim.atoms.write(f'dimer_cluster{c}.pdb')
    view.addModel(open(f'dimer_cluster{c}.pdb').read(), 'pdb')
    view.setStyle({'model': c}, {'cartoon': {'color': colors[c]}})
view.zoomTo()
view.show()

**QUESTIONS❓**

1. How many clusters do you need to describe most of the simulation? Where do the representatives differ?
2. The number of clusters depends on the threshold. Try 1.5 and 2.5 Å: how do the results change? What is a reasonable choice, given the RMSD values you see in the matrix?
3. **Try it:** cluster the monomer trajectory with the same threshold and compare.

In [ ]:
# your code here

### III.3 – Principal Component Analysis (PCA)

PCA decomposes the fluctuations of the atoms into a set of collective motions, the *principal components* (PCs), ordered so that the first one accounts for the largest part of the variance of the atomic positions, the second for the largest part of what remains, and so on. Applied to an MD trajectory, it often shows that a few PCs describe most of the motion: these are the large-scale motions of the protein, sometimes called its *essential dynamics*.

The method:

1. superimpose all the frames (the trajectory is already superimposed);
2. build the $3N \times 3N$ covariance matrix of the coordinates of the $N$ atoms, with respect to their average positions;
3. diagonalise the covariance matrix: the eigenvectors are the PCs and the eigenvalues are their variances;
4. sort the PCs by decreasing variance.

In [ ]:
from MDAnalysis.analysis import pca

pc_dim = pca.PCA(dim, select='backbone', align=False).run()   # align=False: the trajectory is already superimposed
backbone = dim.select_atoms('backbone')

print(f'{backbone.n_atoms} backbone atoms -> {pc_dim.results.p_components.shape[1]} principal components')
print('variance of the first PC (Å²):', pc_dim.results.variance[0])

The variance of a PC is more meaningful as a fraction of the total variance. `cumulated_variance` gives the fraction of the variance described by the first PCs together:

In [ ]:
cumulated = pc_dim.results.cumulated_variance
for i in range(4):
    print(f'first {i + 1} PCs: {cumulated[i] * 100:.1f}% of the variance')

plt.plot(np.arange(1, 21), cumulated[:20], 'o-')
plt.xlabel('Number of principal components')
plt.ylabel('Cumulated variance')
plt.show()

**QUESTION❓:** how many components do you need to describe 90% of the variance? This means that most of the motion in the trajectory can be described with that many numbers instead of the thousands of coordinates of the atoms.

#### Projecting the trajectory on the principal components

The `transform` method projects each frame on the PCs, giving the *weight* (or projection) $w_i(t)$ of PC $i$ at time $t$:

$$w_i(t) = (\mathbf{r}(t) - \bar{\mathbf{r}}) \cdot \mathbf{u}_i$$

where $\mathbf{r}(t)$ are the coordinates at time $t$, $\bar{\mathbf{r}}$ the average coordinates and $\mathbf{u}_i$ the eigenvector of PC $i$. Plotting the first two projections against each other gives a two-dimensional map of the conformations visited by the protein. Here we colour each frame by time and, on the right, by the cluster it belongs to.

In [ ]:
proj_dim = pc_dim.transform(backbone, n_components=3)    # shape (n_frames, n_components)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sc = axes[0].scatter(proj_dim[:, 0], proj_dim[:, 1], c=times, s=5, cmap='viridis')
fig.colorbar(sc, ax=axes[0], label='Time (ns)')
for c in range(min(5, labels_dim.max() + 1)):
    sel = labels_dim == c
    axes[1].scatter(proj_dim[sel, 0], proj_dim[sel, 1], s=5, label=f'cluster {c}')
axes[1].legend(markerscale=3)
for ax in axes:
    ax.set_xlabel('PC1 (Å)')
    ax.set_ylabel('PC2 (Å)')
plt.show()

#### Visualising a principal component

The motion described by PC $i$ is $\bar{\mathbf{r}} + w\,\mathbf{u}_i$, the average structure displaced along the eigenvector. We build a short movie in which $w$ goes from the minimum to the maximum value it has in the simulation and back:

In [ ]:
def pc_movie(pc, atoms, projections, i, n_frames=20):
    """A Universe with the motion along PC i, between the extreme values of its projection."""
    w = np.linspace(projections[:, i].min(), projections[:, i].max(), n_frames)
    w = np.concatenate([w, w[::-1]])
    coordinates = pc.mean + np.outer(w, pc.results.p_components[:, i]).reshape(len(w), -1, 3)
    movie = mda.Merge(atoms)
    movie.load_new(coordinates, order='fac')
    return movie

movie = pc_movie(pc_dim, backbone, proj_dim, 0)
movie.atoms.write('PC1_movie.pdb', frames='all')

view = py3Dmol.view(width=700, height=450)
view.addModelsAsFrames(open('PC1_movie.pdb').read(), 'pdb')
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'purple'}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'orange'}})
view.zoomTo()
view.animate({'loop': 'forward', 'interval': 100})
view.show()

You can also look at the movie with VMD on your laptop: download `PC1_movie.pdb` and open it. To see it as a cartoon, use `Graphics -> Representations` and choose `NewCartoon`.

**QUESTIONS❓**

1. Which parts of the dimer move along PC1? Is it a motion of one chain with respect to the other, or a motion inside the chains?
2. Look at PC2 and PC3 as well (change the last argument of `pc_movie`).

In [ ]:
# your code here

### III.4 – Free-energy landscape

If the simulation samples the conformations according to their equilibrium probability, the probability $P(w_1, w_2)$ of finding the protein at a point of the PC1-PC2 map is related to the free energy:

$$F(w_1, w_2) = -k_BT \ln P(w_1, w_2)$$

The minima of $F$ are the most populated conformational states, and the barriers between them tell how difficult it is to go from one to the other. We estimate $P$ with a two-dimensional histogram, assuming a temperature of 300 K.

In [ ]:
kT = 0.008314 * 300    # kJ/mol

def free_energy(x, y, bins=15):
    histogram, x_edges, y_edges = np.histogram2d(x, y, bins=bins, density=True)
    with np.errstate(divide='ignore'):
        F = -kT * np.log(histogram)
    F -= F[np.isfinite(F)].min()           # the minimum is set to zero
    return F, x_edges, y_edges

F, x_edges, y_edges = free_energy(proj_dim[:, 0], proj_dim[:, 1])
plt.figure(figsize=(7, 5.5))
plt.contourf((x_edges[:-1] + x_edges[1:]) / 2, (y_edges[:-1] + y_edges[1:]) / 2, F.T, levels=15, cmap='viridis')
plt.colorbar(label='Free energy (kJ/mol)')
plt.xlabel('PC1 (Å)')
plt.ylabel('PC2 (Å)')
plt.show()

**QUESTIONS❓**

1. How many minima do you see? Do they correspond to the clusters found in III.2?
2. The white regions were never visited. Does it mean that their free energy is infinite? How reliable is a free energy estimated from 1000 frames of a 20 ns simulation?

### III.5 – Is the sampling converged?

All the analyses above assume that the simulation is long enough to represent the motions of the protein. A simple test is to repeat the analysis on two halves of the trajectory and check whether the results agree. For PCA, the agreement between two sets of PCs $\mathbf{u}_i$ and $\mathbf{v}_j$ is measured by the *root mean square inner product* (RMSIP) of the first $n$ components:

$$RMSIP = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\sum_{j=1}^{n} (\mathbf{u}_i \cdot \mathbf{v}_j)^2}$$

RMSIP is 1 if the two sets describe the same motions (even in a different order) and 0 if the motions are completely different.

In [ ]:
n_half = dim.trajectory.n_frames // 2
pc_first = pca.PCA(dim, select='backbone', align=False).run(stop=n_half)
pc_second = pca.PCA(dim, select='backbone', align=False).run(start=n_half)

print(f'RMSIP of the first 10 PCs between the two halves: {pc_first.rmsip(pc_second, n_components=10):.2f}')

plt.plot(np.arange(1, 21), pc_first.results.cumulated_variance[:20], 'o-', label='first half')
plt.plot(np.arange(1, 21), pc_second.results.cumulated_variance[:20], 'o-', label='second half')
plt.plot(np.arange(1, 21), cumulated[:20], 'k--', label='whole trajectory')
plt.xlabel('Number of principal components')
plt.ylabel('Cumulated variance')
plt.legend()
plt.show()

**QUESTIONS❓**

1. Do the two halves describe the same motions? How would you expect the RMSIP to change with a longer simulation, or with several independent simulations (replicas) like those of the previous task?
2. **Try it:** compute the RMSIP between the PCs of the first and the second half of the monomer trajectory, and between the PCs of chain A and chain B of the dimer (select the backbone of one chain and superimpose on that chain). What do you learn?

In [ ]:
# your code here

## What next

In the report you will apply these analyses to a new system. Many other analyses are available in MDAnalysis: see the [user guide](https://userguide.mdanalysis.org/stable/examples/analysis/README.html).